# Fuel Invoice data processing 
This file goes through the process of reading the pre-processed textract APIs csv output file, extracting and transforming the data and saving the data into the target S3 folder in the parquet format partitioned by year and month. The pandas library has been used for faster delivery as it is a small file. In production this can be replicated using the spark dataframe for faster processing of larger files

Importing the libraries

In [80]:
import pandas as pd
import numpy as np
import boto3

Initialising the schema that is expected at the warehouse layer and accessing the files from S3 input path

In [81]:
final = pd.DataFrame()
dfs = []
schema = ['city', 'state', 'name', 'invoice_receipt_date', 'invoice_receipt_id', 'tax_payer_id', 'total', 'time', 'litres', 'pricelitre', 'type', 'product', 'grade',
          'fhst', 'phst', 'gst']


Function to perform the transformation for data cleaning

In [82]:
def transformation_process(file_path):
    df = pd.read_csv(file_path)
    
    # Removing undesired symbols and cleaning the data
    df["'Value"] = df["'Value"].replace({"^'":""}, regex=True)
    
    df["'Normalized"] = df["'Normalized"].replace({"^'":""}, regex=True)
    df["'Normalized"] = df["'Normalized"].str.lower()

    df["'Key"] = df["'Key"].replace(r'[^a-zA-Z]',"", regex=True)
    df["'Key"] = df["'Key"].str.lower()

    print("File processing: " + key)
    
    
    # initialising list of keys needed to extract from the invoice
    tax_values = ['gstincl','gstincluded','fhstincludedinfuel','fhstincl','phstincludedinfuel','phstincl']

    other_values= ['time','product','litres','grade','pricelitre','type']

    normal_values = ['city','state','name','invoice_receipt_date','invoice_receipt_id','total','tax_payer_id']
    
    # Calling function to extract correct column names
    new_columns = df.apply(col_func, axis=1)

    new_row = df["'Value"].to_list()
    # print(new_columns)


    # Removing undesired symbols from the column names 
    new_columns = new_columns.str.lower()
    new_columns = new_columns.str.replace("'","")
    new_columns = new_columns.str.replace("#","")
    new_columns = new_columns.str.replace(" :","")
    new_columns = new_columns.str.replace(":","")


    # Adding the data as a row and columns to the new dataframe
    data = pd.DataFrame([new_row], columns = new_columns)
    
    
    data = data.loc[:,~data.columns.duplicated(keep='first')]


    data.columns = ['gst' if 'gst' in org_column_name else org_column_name for org_column_name in data.columns]
    data.columns = ['fhst' if 'fhst' in org_column_name else org_column_name for org_column_name in data.columns]
    data.columns = ['phst' if 'phst' in org_column_name else org_column_name for org_column_name in data.columns]
    
    
    data = schema_check(data)
    
            
    # data = data.loc[:,~data.columns.duplicated(keep='first')]

    data =data[schema]

    dfs.append(data)
    print("File appended")


Extracting meaningful column names from the csv

In [83]:

def col_func(row):
    if row["'Normalized"]=="other" :
        if row["'Key"] in other_values:
            value = row["'Key"]
        else:
            value = "ignore"
    elif row["'Normalized"]=="tax":
        val_check = row["'Key"]
        if val_check in tax_values:
            value = row["'Key"]
        else:
            value = "ignore"

    elif row["'Normalized"] in normal_values:
        value = row["'Normalized"]
    else:
        value = "ignore"
    return value
    


Schema check to ensure it is as per the warehouse layer design

In [84]:
def schema_check(data):
    for col in data.columns:
                # print(col)
        if col not in schema:
            data = data.drop(columns=col)

    for col in schema:
        if col not in data.columns:
            data[col] = pd.NA
    return data

Reading files from S3 and processing

In [85]:
s3 = boto3.resource('s3')
bucket = s3.Bucket('cn01-project-pre-processed-files-205096516800-us-east-2-an')
for obj in bucket.objects.filter(Prefix= "input_files/invoice_files"):
    key = obj.key
    print(key)
    if key == "input_files/invoice_files/":
        continue
    

    file_path = "s3://" + bucket.name + "/" + str(key)
    transformation_process(file_path)
    
    
    

input_files/invoice_files/
input_files/invoice_files/summaryFields-1.csv
File processing: input_files/invoice_files/summaryFields-1.csv
File appended
input_files/invoice_files/summaryFields-10.csv
File processing: input_files/invoice_files/summaryFields-10.csv
File appended
input_files/invoice_files/summaryFields-11.csv
File processing: input_files/invoice_files/summaryFields-11.csv
File appended
input_files/invoice_files/summaryFields-2.csv
File processing: input_files/invoice_files/summaryFields-2.csv
File appended
input_files/invoice_files/summaryFields-3.csv
File processing: input_files/invoice_files/summaryFields-3.csv
File appended
input_files/invoice_files/summaryFields-4.csv
File processing: input_files/invoice_files/summaryFields-4.csv
File appended
input_files/invoice_files/summaryFields-5.csv
File processing: input_files/invoice_files/summaryFields-5.csv
File appended
input_files/invoice_files/summaryFields-6.csv
File processing: input_files/invoice_files/summaryFields-6.csv

Performing data formatting and ensuring undesired symbols are removed from values

In [86]:

final = pd.concat(dfs)

final['invoice_receipt_date'] = pd.to_datetime(final['invoice_receipt_date'])
final['year'] = final['invoice_receipt_date'].dt.year
final['month'] = final['invoice_receipt_date'].dt.month
final['invoice_receipt_date'] = pd.to_datetime(final['invoice_receipt_date'], format='ISO8601').dt.date


final['total'] = final['total'].str.extract(r'(\d+[.\d]*)')
final['litres'] = final['litres'].str.extract(r'(\d+[.\d]*)')
final['pricelitre'] = final['pricelitre'].str.extract(r'(\d+[.\d]*)')
final['fhst'] = final['fhst'].str.extract(r'(\d+[.\d]*)')
final['phst'] = final['phst'].str.extract(r'(\d+[.\d]*)')
final['gst'] = final['gst'].str.extract(r'(\d+[.\d]*)')


final['total'] = final['total'].astype(float)
final['litres'] = final['litres'].astype(float)
final['pricelitre'] = final['pricelitre'].astype(float)
final['fhst'] = final['fhst'].astype(float)
final['phst'] = final['phst'].astype(float)
final['gst'] = final['gst'].astype(float)

final["city"] = final["city"].replace(r'[^a-z A-Z]',"", regex=True)
city_map = {"Kelouna":"Kelowna","Winnipog":"Winnipeg"}
final['city'] = final['city'].replace(city_map)
final['city'] = final['city'].str.lower()

pd.set_option('display.max_columns', None)


# Dictionary mapping to get the province for each record
city_state_map = {"edmonton":"AB","caledon":"ON","willowdale":"ON","kelowna":"BC","langley":"BC","brooks":"BC","burnaby":"BC","brandon":"ON","red deer":"AB","saskatoon":"SK","consort":"AB",
"prince albert":"SK","barrie":"ON","regina":"SK","winnipeg":"MB","kingston":"ON","thunder bay":"ON","kamloops":"BC","nisku":"AB","hanna":"AB","golden west":"BC"}
final['state'] = final['city'].map(city_state_map)

final['city'] = final['city'].str.lower().str.strip()

print(final.head(5))



         city state                 name invoice_receipt_date  \
0       hanna    AB            UFA Hanna           2022-08-10   
0    edmonton    AB         PETRO-CANADA           2016-06-03   
0     kelowna    BC  Car"Northgate Frev.           2023-01-07   
0    winnipeg    MB               costco           2025-11-30   
0  willowdale    ON         PETRO-CANADA           2023-06-21   

  invoice_receipt_id tax_payer_id  total      time   litres  pricelitre  \
0       50746215NSYX         <NA>    NaN  07:46:18  266.200         NaN   
0               <NA>               55.16      <NA>      NaN         NaN   
0             000416   R743318321  45.42      <NA>      NaN         NaN   
0       # 0010010170   #121476329  41.03      <NA>   35.313       1.179   
0             075795    874383177  30.00  13:13:37      NaN         NaN   

       type product     grade  fhst  phst   gst  year  month  
0      <NA>  DIESEL      <NA>   NaN   NaN   NaN  2022      8  
0      <NA>    <NA>      <NA>   

In [87]:
final.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11 entries, 0 to 0
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   city                  11 non-null     object 
 1   state                 11 non-null     object 
 2   name                  11 non-null     object 
 3   invoice_receipt_date  11 non-null     object 
 4   invoice_receipt_id    9 non-null      object 
 5   tax_payer_id          7 non-null      object 
 6   total                 9 non-null      float64
 7   time                  4 non-null      object 
 8   litres                3 non-null      float64
 9   pricelitre            1 non-null      float64
 10  type                  2 non-null      object 
 11  product               2 non-null      object 
 12  grade                 1 non-null      object 
 13  fhst                  2 non-null      float64
 14  phst                  2 non-null      float64
 15  gst                   4 no

Writing file to target as a parquet partitioned file

In [15]:
# final.to_parquet(path="s3://cn01-project-output-205096516800-us-east-2-an/output_parquet_files/fuel_invoice/",partition_cols=['year','month'])